#   LangGraph 활용 - Multi-Agent 아키텍처

---

## 환경 설정 및 준비

`(1) Env 환경변수`

In [ ]:
from dotenv import load_dotenv
load_dotenv()

`(2) 기본 라이브러리`

In [ ]:
import os
from glob import glob

from pprint import pprint
import json

import warnings
warnings.filterwarnings("ignore")

`(3) Langsmith tracing 설정`

In [ ]:
# Langsmith tracing 여부를 확인 (true: langsmith 추척 활성화, false: langsmith 추척 비활성화)
import os
print(os.getenv('LANGSMITH_TRACING'))

---

## 2. 기본 개념

### 2.1 Multi-Agent System 이해

- **1. 에이전트란?**
  - **에이전트**는 LLM을 사용하여 애플리케이션의 제어 흐름을 결정하는 시스템

- **2. 멀티 에이전트 시스템이 필요한 이유**: 단일 에이전트 시스템이 복잡해지면서 다음과 같은 문제가 발생
  - **도구 과부하**: 에이전트가 너무 많은 도구를 가져 잘못된 결정을 내림
  - **컨텍스트 복잡성**: 단일 에이전트가 추적하기에 너무 복잡한 컨텍스트
  - **전문화 필요**: 플래너, 연구자, 수학 전문가 등 여러 전문 영역이 필요

- **3. 멀티 에이전트 시스템의 주요 장점**
  - **모듈성**: 개별 에이전트로 분리하여 개발, 테스트, 유지보수가 용이
  - **전문화**: 특정 도메인에 초점을 맞춘 전문 에이전트 생성으로 전체 성능 향상
  - **제어**: 에이전트 간 통신을 명시적으로 제어 가능

In [ ]:
"""
Multi-Agent System (MAS)
- 여러 AI 에이전트가 협업하여 복잡한 작업 해결
- 각 에이전트는 특정 전문 영역 담당
- LangGraph에서는 각 에이전트가 그래프의 노드로 표현
"""

from typing import TypedDict, Annotated, Literal, List
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, MessagesState, START, END

# 기본 State 정의
class AgentState(MessagesState):
    """멀티에이전트 시스템의 공유 상태"""
    current_agent: str
    task_status: str
    results: dict

### 2.2 Command 타입 이해

In [ ]:
from langgraph.types import Command

def example_agent_node(state: AgentState) -> Command:
    """Command를 사용한 에이전트 노드 예제"""
    
    # 작업 수행
    result = "작업 완료"
    
    # Command 반환 - 다음 노드와 상태 업데이트 지정
    return Command(
        goto="next_agent",  # 다음 노드
        update={            # 상태 업데이트
            "messages": state["messages"] + [AIMessage(content=result)],
            "task_status": "completed"
        },
        graph=Command.PARENT  # 부모 그래프에서 실행
    )

**[예제] 멀티 에이전트 간 네비게이션**

- 서브그래프 간에 제어권을 넘길 수 있음

In [ ]:
from langgraph.types import Command

# 에이전트 A 서브그래프
def agent_a_node(state: dict):
    if state.get("need_help"):
        # 에이전트 B로 제어권 넘김
        return Command(
            goto="agent_b",
            update={"message": "도움이 필요합니다"},
            graph=Command.PARENT  # 부모 그래프로 이동
        )
    return {"result": "A 작업 완료"}

# 에이전트 B 서브그래프  
def agent_b_node(state: dict):
    return {"result": "B가 도움을 제공했습니다: " + state.get("message", "")}

# 서브그래프들 생성
agent_a_builder = StateGraph(dict)
agent_a_builder.add_node("work", agent_a_node)
agent_a_builder.add_edge(START, "work")
agent_a = agent_a_builder.compile()

agent_b_builder = StateGraph(dict)
agent_b_builder.add_node("help", agent_b_node)  
agent_b_builder.add_edge(START, "help")
agent_b = agent_b_builder.compile()

# 멀티 에이전트 부모 그래프
multi_agent_builder = StateGraph(dict)
multi_agent_builder.add_node("agent_a", agent_a)
multi_agent_builder.add_node("agent_b", agent_b)

multi_agent_builder.add_edge(START, "agent_a")
multi_agent_graph = multi_agent_builder.compile()

# 실행
result = multi_agent_graph.invoke({"need_help": True})
print("멀티 에이전트 결과:", result)

---

## 3. Supervisor 패턴

- **특징**: 단일 슈퍼바이저 에이전트가 다른 에이전트들의 실행을 결정
- **적용**: 중앙 집중식 제어가 필요한 경우

![Supervisor 패턴](https://langchain-ai.github.io/langgraph/agents/assets/supervisor.png)

### 3.1 기본 Supervisor 구현

`(1) langgraph-supervisor 패키지 사용`

- **설치**

    ```bash
    pip install langgraph-supervisor 
    ```

    ```bash
    uv add langgraph-supervisor
    ```

In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_tavily import TavilySearch
from langchain_core.tools import tool
from IPython.display import Image, display
from langchain_openai import ChatOpenAI

# 1. 작업자 에이전트들 생성

# 연구 에이전트
tavily_search = TavilySearch(max_results=3)
research_agent = create_react_agent(
    model=ChatOpenAI(model="gpt-4.1-mini"),
    tools=[tavily_search],
    name="research_agent",
    prompt="""당신은 연구 전문가입니다.
    웹 검색을 통해 정확하고 최신 정보를 찾아 제공하세요.
    검색 결과를 요약하여 핵심 정보를 전달하세요."""
)

# 수학 에이전트
@tool
def calculate(expression: str) -> str:
    """수학 계산을 수행합니다.
    
    Args:
        expression: 계산할 수식 (예: "2 + 2", "10 * 5")
    """
    try:
        result = eval(expression)
        return f"계산 결과: {result}"
    except:
        return "계산할 수 없는 수식입니다."

math_agent = create_react_agent(
    model=ChatOpenAI(model="gpt-4.1-mini"),
    tools=[calculate],
    name="math_agent",
    prompt="""당신은 수학 전문가입니다. 
    주어진 수학 문제를 정확하게 계산하고 설명하세요.
    계산 도구를 사용하여 정확한 답을 제공하세요."""
)

# 2. 감독자 시스템 생성
from langgraph_supervisor import create_supervisor

supervisor_app = create_supervisor(
    model=ChatOpenAI(model="gpt-4.1"),
    agents=[research_agent, math_agent],
    prompt="""당신은 팀 관리자입니다. 사용자 요청을 분석하여:
    - 수학 계산이 필요하면 → math_expert
    - 정보 검색이 필요하면 → research_expert
    
    적절한 전문가에게 작업을 할당하세요."""
).compile()
    
# 그래프 시각화
# display(Image(supervisor_app.get_graph().draw_mermaid()))

In [ ]:
from langchain_core.runnables.graph import CurveStyle
from IPython.display import Markdown, display

mermaid_code = supervisor_app.get_graph().draw_mermaid()
display(Markdown(f"```mermaid\n{mermaid_code}\n```"))

In [ ]:
mermaid_code = supervisor_app.get_graph(xray=True).draw_mermaid()
display(Markdown(f"```mermaid\n{mermaid_code}\n```"))

In [ ]:
# 3. 실행
queries = [
    "한국의 인구는 몇 명인가요?",
    "1234 * 5678을 계산해주세요.",
    "파이썬의 창시자는 누구이고, 100 + 200은 얼마인가요?"
]

for query in queries:
    print(f"\n📝 질문: {query}")
    print("-" * 50)
    
    result = supervisor_app.invoke({
        "messages": [{"role": "user", "content": query}]
    })
    
    # 메시지 출력
    for m in result["messages"]:
        m.pretty_print()
    print("\n\n")

### 3.2 Handoff Tool을 사용한 Supervisor 패턴 구현

- **핸드오프 도구**를 사용하여 에이전트 간 통신을 명시적으로 제어 
- 현재 에이전트에서 다음 에이전트로 이동하는 데 사용되는 도구

In [ ]:
from langgraph.graph import MessagesState, StateGraph, START, END
from langgraph_supervisor import create_handoff_tool

# Supervisor → 작업자 핸드오프
to_researcher = create_handoff_tool(
    agent_name="researcher", 
    description="연구 작업을 연구원에게 할당"
)
to_analyst = create_handoff_tool(
    agent_name="analyst",
    description="분석 작업을 분석가에게 할당"
)

# 작업자 → Supervisor 복귀
to_supervisor = create_handoff_tool(
    agent_name="supervisor",
    description="작업 완료 후 감독자에게 보고"
)

# Supervisor 에이전트
supervisor = create_react_agent(
    model=ChatOpenAI(model="gpt-4.1"),
    tools=[to_researcher, to_analyst],
    name="supervisor",
    prompt="""당신은 프로젝트 감독자입니다.
    작업을 적절한 팀원에게 분배하세요:
    - 정보 수집 → researcher
    - 데이터 분석 → analyst"""
)

# Researcher 에이전트
researcher = create_react_agent(
    model=ChatOpenAI(model="gpt-4.1-mini"),
    tools=[tavily_search, to_analyst, to_supervisor],
    name="researcher",
    prompt="""당신은 연구원입니다.
    정보를 검색하고 수집합니다.
    분석이 필요하면 analyst에게, 완료되면 supervisor에게 보고하세요."""
)

# Analyst 에이전트
analyst = create_react_agent(
    model=ChatOpenAI(model="gpt-4.1-mini"),
    tools=[calculate, to_supervisor],
    name="analyst",
    prompt="""당신은 데이터 분석가입니다.
    데이터를 분석하고 인사이트를 도출합니다.
    작업 완료 후 supervisor에게 보고하세요."""
)
# 그래프 구성
supervisor_graph= (
    StateGraph(MessagesState)
    .add_node("supervisor", supervisor)
    .add_node("researcher", researcher)
    .add_node("analyst", analyst)
    .add_edge(START, "supervisor")  # 항상 슈퍼바이저부터 시작
    .compile()
)

In [ ]:
# 그래프 구성 - (시각화 목적)
supervisor_app = (
    StateGraph(MessagesState)
    .add_node("supervisor", supervisor)
    .add_node("researcher", researcher)
    .add_node("analyst", analyst)
    .add_edge(START, "supervisor")
    
    # Supervisor → Workers
    .add_edge("supervisor", "researcher")
    .add_edge("supervisor", "analyst")
    
    # Workers → Supervisor (복귀)
    .add_edge("researcher", "supervisor")
    .add_edge("analyst", "supervisor")
    
    # Workers 간 협업
    .add_edge("researcher", "analyst")
    
    .compile()
)

# 그래프 시각화
# display(supervisor_app.get_graph().draw_mermaid())

In [ ]:
from langchain_core.runnables.graph import CurveStyle
from IPython.display import Markdown, display

mermaid_code = supervisor_app.get_graph().draw_mermaid()
display(Markdown(f"```mermaid\n{mermaid_code}\n```"))

In [ ]:
# 실행
query = "파이썬의 창시자는 누구이고, 100 + 200은 얼마인가요?"
result = supervisor_graph.invoke({
   "messages": [HumanMessage(content=query)]
})

for m in result["messages"]:
    m.pretty_print()

---

## 4. Swarm 패턴

- **특징**: 분산형, 에이전트 간 자율적 협력
- **설치**
    - langgraph-swarm 패키지 사용

    ```bash
    pip install langgraph-swarm 
    ```

    ```bash
    uv add langgraph-swarm
    ```

![Swarm 패턴](https://langchain-ai.github.io/langgraph/agents/assets/swarm.png)


In [ ]:
from langgraph_swarm import create_swarm

# 핸드오프 도구 생성
transfer_to_math_agent = create_handoff_tool(
    agent_name="math_agent",
    description="수학 계산이 필요할 때 수학 전문가에게 전달합니다."
)

transfer_to_research_agent = create_handoff_tool(
    agent_name="research_agent", 
    description="정보 검색이나 조사가 필요할 때 연구 전문가에게 전달합니다."
)

# 연구 에이전트 (핸드오프 도구 포함)
research_agent = create_react_agent(
    model=ChatOpenAI(model="gpt-4.1-mini"),
    tools=[tavily_search, transfer_to_math_agent],
    prompt="""당신은 연구 전문가입니다. 웹 검색을 통해 정보를 찾아 제공합니다.
    
만약 검색 결과에서 숫자 데이터를 찾았고 사용자가 계산을 요청한다면, 
transfer_to_math_agent 도구를 사용해서 수학 전문가에게 작업을 전달하세요.""",
    name="research_agent"
)

# 수학 에이전트 (핸드오프 도구 포함)
math_agent = create_react_agent(
    model=ChatOpenAI(model="gpt-4.1-mini"),
    tools=[calculate, transfer_to_research_agent],
    prompt="""당신은 수학 전문가입니다. 정확한 계산을 수행합니다.
    
만약 계산에 필요한 데이터가 없거나 추가 정보가 필요하다면,
transfer_to_research_agent 도구를 사용해서 연구 전문가에게 작업을 전달하세요.""",
    name="math_agent"
)

# 스웜 생성
swarm = create_swarm(
    agents=[research_agent, math_agent],
    default_active_agent="research_agent"  # 기본적으로 연구 에이전트가 먼저 시작
).compile()

# 그래프 시각화
# display(swarm.get_graph().draw_mermaid())

In [ ]:
from langchain_core.runnables.graph import CurveStyle
from IPython.display import Markdown, display

mermaid_code = swarm.get_graph().draw_mermaid()
display(Markdown(f"```mermaid\n{mermaid_code}\n```"))

In [ ]:
# 실행
query = "파이썬의 창시자는 누구이고, 100 + 200은 얼마인가요?"
result = swarm.invoke({
   "messages": [HumanMessage(content=query)]
})

for m in result["messages"]:
    m.pretty_print()

In [ ]:
from langgraph_swarm import create_swarm
from langgraph.checkpoint.memory import InMemorySaver

# 상호 핸드오프 도구
to_planner = create_handoff_tool(
    agent_name="planner",
    description="계획 수립이 필요할 때 플래너에게 전달"
)

to_executor = create_handoff_tool(
    agent_name="executor",
    description="실행이 필요할 때 실행자에게 전달"
)

to_reviewer = create_handoff_tool(
    agent_name="reviewer",
    description="검토가 필요할 때 검토자에게 전달"
)

# Planner 에이전트
planner = create_react_agent(
    model=ChatOpenAI(model="gpt-4.1-mini"),
    tools=[to_executor, to_reviewer],
    name="planner",
    prompt="""당신은 계획 수립 전문가입니다.
    작업을 단계별로 계획하고, 실행자에게 전달하세요.
    복잡한 작업은 검토자와 상의하세요."""
)

# Executor 에이전트
executor = create_react_agent(
    model=ChatOpenAI(model="gpt-4.1-mini"),
    tools=[tavily_search, calculate, to_reviewer, to_planner],
    name="executor",
    prompt="""당신은 실행 전문가입니다.
    계획된 작업을 실행하고, 필요시 도구를 사용하세요.
    결과는 검토자에게 전달하거나, 추가 계획이 필요하면 플래너에게 문의하세요."""
)

# Reviewer 에이전트
reviewer = create_react_agent(
    model=ChatOpenAI(model="gpt-4.1-mini"),
    tools=[to_planner, to_executor],
    name="reviewer",
    prompt="""당신은 품질 검토 전문가입니다.
    작업 결과를 검토하고 피드백을 제공하세요.
    수정이 필요하면 적절한 에이전트에게 재작업을 요청하세요."""
)

# Swarm 생성
swarm_app = create_swarm(
    agents=[planner, executor, reviewer],
    default_active_agent="planner"  # 기본적으로 플래너 에이전트 먼저 시작
).compile(checkpointer=InMemorySaver())


# 그래프 시각화
# display(swarm_app.get_graph().draw_mermaid_png())

In [ ]:
from langchain_core.runnables.graph import CurveStyle
from IPython.display import Markdown, display

mermaid_code = swarm_app.get_graph().draw_mermaid()
display(Markdown(f"```mermaid\n{mermaid_code}\n```"))

In [ ]:
# 대화형 실행 
config = {"configurable": {"thread_id": "swarm_001"}}

query = "다음 주 프레젠테이션을 준비해야 합니다. 주제는 'AI의 미래'입니다."
result = swarm_app.invoke(
    {"messages": [HumanMessage(content=query)]},
    config
)
for m in result["messages"]:
    m.pretty_print()

### **3. 계층적 아키텍처 (Supervisor + Swarm 조합)** 

- **특징**: 슈퍼바이저와 스웜을 조합하여 복잡한 작업 흐름 구현
- **적용**: 대규모 시스템에서 에이전트 팀들을 관리할 때 (상위 슈퍼바이저가 하위 스웜들을 관리하는 계층적 시스템)

In [ ]:
from langgraph_supervisor import create_supervisor, create_handoff_tool
from langgraph_swarm import create_swarm
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool

# 하위 스웜 1: 정보 수집 팀
to_analyst = create_handoff_tool(
    agent_name="data_analyst",
    description="상세한 데이터 분석이 필요할 때 데이터 분석가에게 전달"
)

basic_researcher = create_react_agent(
    model=ChatOpenAI(model="gpt-4.1-mini"),
    tools=[tavily_search, to_analyst],
    prompt="기본 연구자. 웹 검색으로 정보 수집. 상세 분석이 필요하면 데이터 분석가에게 전달.",
    name="basic_researcher"
)

to_researcher = create_handoff_tool(
    agent_name="basic_researcher", 
    description="기본 정보 검색이 필요할 때 기본 연구자에게 전달"
)

data_analyst = create_react_agent(
    model=ChatOpenAI(model="gpt-4.1-mini"),
    tools=[tavily_search, to_researcher],
    prompt="데이터 분석가. 상세한 분석과 인사이트 제공. 기본 검색이 필요하면 기본 연구자에게 전달.",
    name="data_analyst"
)

research_swarm = create_swarm(
    agents=[basic_researcher, data_analyst],
    default_active_agent="basic_researcher"
).compile(name="research_swarm")

# 하위 스웜 2: 계산 팀  
to_advanced_calculator = create_handoff_tool(
    agent_name="advanced_calculator",
    description="복잡한 계산이 필요할 때 고급 계산기에게 전달"
)

basic_calculator = create_react_agent(
    model=ChatOpenAI(model="gpt-4.1-mini"),
    tools=[calculate, to_advanced_calculator],
    prompt="기본 계산기. 간단한 덧셈과 곱셈 수행. 복잡한 계산은 고급 계산기에게 전달.",
    name="basic_calculator"
)

to_basic_calculator = create_handoff_tool(
    agent_name="basic_calculator",
    description="기본 계산이 필요할 때 기본 계산기에게 전달"
)

advanced_calculator = create_react_agent(
    model=ChatOpenAI(model="gpt-4.1"),
    tools=[calculate, to_basic_calculator],
    prompt="고급 계산기. 복잡한 계산 수행. 기본 계산은 기본 계산기에게 전달.",
    name="advanced_calculator"
)

calc_swarm = create_swarm(
    agents=[basic_calculator, advanced_calculator],
    default_active_agent="basic_calculator"
).compile(name="calc_swarm")

# 최상위 슈퍼바이저
top_supervisor = create_supervisor(
    agents=[research_swarm, calc_swarm],
    model=ChatOpenAI(model="gpt-4.1"),
    prompt="""최상위 슈퍼바이저입니다. 두 개의 전문 팀을 관리합니다:

1. research_swarm: 정보 검색과 데이터 분석 담당
2. calc_swarm: 수학 계산 담당

작업의 성격에 따라 적절한 팀에게 할당하세요."""
).compile(name="top_supervisor")


# 그래프 시각화
# display(Image(top_supervisor.get_graph(xray=False).draw_mermaid_png()))

In [ ]:
from langchain_core.runnables.graph import CurveStyle
from IPython.display import Markdown, display

mermaid_code = top_supervisor.get_graph().draw_mermaid()
display(Markdown(f"```mermaid\n{mermaid_code}\n```"))

In [ ]:
from langchain_core.runnables.graph import CurveStyle
from IPython.display import Markdown, display

mermaid_code = top_supervisor.get_graph(xray=True).draw_mermaid()
display(Markdown(f"```mermaid\n{mermaid_code}\n```"))

In [ ]:
# 실행
result = top_supervisor.invoke({
    "messages": [{
        "role": "user", 
        "content": "한국의 인구를 찾아서, 인구 수에 2를 곱해주세요."
    }]
})

for m in result["messages"]:
    m.pretty_print()

### 4.2 동적 Swarm with State

In [ ]:
from typing import TypedDict, List
import uuid

class SwarmState(MessagesState):
    """Swarm 상태 정의"""
    active_agent: str
    task_queue: List[dict]
    completed_tasks: List[dict]
    agent_history: List[str]
    iteration: int

class DynamicSwarm:
    """상태를 관리하는 동적 Swarm"""
    
    def __init__(self):
        self.agents = {}
        self.setup_dynamic_agents()
        self.build_swarm_graph()
    
    def create_agent_with_state(self, name: str, speciality: str):
        """상태 인식 에이전트 생성"""
        
        def agent_node(state: SwarmState) -> Command:
            # 현재 상태 분석
            iteration = state.get("iteration", 0)
            task_queue = state.get("task_queue", [])
            
            # 작업 처리
            if task_queue:
                current_task = task_queue[0]
                result = f"{name} 처리 완료: {current_task['description']}"
                
                # 다음 에이전트 결정
                if iteration < 3 and len(task_queue) > 1:
                    next_agent = self.select_next_agent(name, task_queue[1])
                else:
                    next_agent = END
                
                return Command(
                    goto=next_agent,
                    update={
                        "messages": state["messages"] + [
                            AIMessage(content=result, name=name)
                        ],
                        "active_agent": next_agent,
                        "task_queue": task_queue[1:],
                        "completed_tasks": state.get("completed_tasks", []) + [current_task],
                        "agent_history": state.get("agent_history", []) + [name],
                        "iteration": iteration + 1
                    }
                )
            
            return Command(goto=END)
        
        self.agents[name] = agent_node
        return agent_node
    
    def select_next_agent(self, current: str, next_task: dict) -> str:
        """작업 유형에 따른 다음 에이전트 선택"""
        task_type = next_task.get("type", "")
        
        agent_mapping = {
            "research": "researcher",
            "analysis": "analyst", 
            "review": "reviewer",
            "execute": "executor"
        }
        
        return agent_mapping.get(task_type, "executor")
    
    def setup_dynamic_agents(self):
        """동적 에이전트 설정"""
        agent_configs = [
            ("researcher", "정보 수집 및 조사"),
            ("analyst", "데이터 분석 및 인사이트 도출"),
            ("executor", "작업 실행 및 구현"),
            ("reviewer", "품질 검토 및 피드백")
        ]
        
        for name, speciality in agent_configs:
            self.create_agent_with_state(name, speciality)
    
    def build_swarm_graph(self):
        """Swarm 그래프 구축"""
        graph = StateGraph(SwarmState)
        
        # 모든 에이전트 노드 추가
        for name, node in self.agents.items():
            graph.add_node(name, node)
        
        # 시작점 설정
        graph.add_edge(START, "researcher")
        
        # 동적 엣지 (각 에이전트는 Command로 다음 노드 지정)
        self.graph = graph.compile()
    
    def run_with_tasks(self, tasks: List[dict]):
        """작업 목록으로 실행"""
        initial_state = {
            "messages": [HumanMessage(content="작업을 시작합니다.")],
            "task_queue": tasks,
            "completed_tasks": [],
            "agent_history": [],
            "active_agent": "researcher",
            "iteration": 0
        }
        
        result = self.graph.invoke(initial_state)
        
        print("\n📊 작업 완료 보고서")
        print("=" * 50)
        print(f"완료된 작업: {len(result['completed_tasks'])}개")
        print(f"참여 에이전트: {' → '.join(result['agent_history'])}")
        
        for task in result['completed_tasks']:
            print(f"  ✓ {task['description']}")

# 실행
dynamic_swarm = DynamicSwarm()

tasks = [
    {"id": "1", "type": "research", "description": "AI 트렌드 조사"},
    {"id": "2", "type": "analysis", "description": "수집된 데이터 분석"},
    {"id": "3", "type": "review", "description": "분석 결과 검토"},
    {"id": "4", "type": "execute", "description": "최종 보고서 작성"}
]

dynamic_swarm.run_with_tasks(tasks)

In [ ]:



```python

```

---

## 5. 계층적 (Hierarchical) 아키텍처

### 5.1 다층 Supervisor 시스템

```python
class HierarchicalSystem:
    """계층적 멀티에이전트 시스템"""
    
    def __init__(self):
        self.llm = ChatOpenAI(model="gpt-4o-mini")
        self.setup_teams()
        self.build_hierarchy()
    
    def setup_teams(self):
        """팀별 에이전트 설정"""
        
        # Research Team
        self.research_team = self.create_research_team()
        
        # Engineering Team  
        self.engineering_team = self.create_engineering_team()
        
        # Top-level Supervisor
        self.top_supervisor = self.create_top_supervisor()
    
    def create_research_team(self):
        """연구팀 생성"""
        
        # 연구팀 멤버
        data_collector = create_react_agent(
            model=self.llm,
            tools=[search_web],
            name="data_collector",
            prompt="데이터 수집 전문가입니다. 웹에서 정보를 수집합니다."
        )
        
        data_analyst = create_react_agent(
            model=self.llm,
            tools=[calculate],
            name="data_analyst",
            prompt="데이터 분석 전문가입니다. 수집된 데이터를 분석합니다."
        )
        
        # 연구팀 감독자
        research_supervisor = create_supervisor(
            agents=[data_collector, data_analyst],
            model=self.llm,
            prompt="""연구팀 감독자입니다.
            - 데이터 수집 → data_collector
            - 데이터 분석 → data_analyst
            팀원들의 작업을 조율합니다."""
        )
        
        return research_supervisor.compile(name="research_team")
    
    def create_engineering_team(self):
        """엔지니어링팀 생성"""
        
        # 엔지니어링팀 멤버
        backend_dev = create_react_agent(
            model=self.llm,
            tools=[],  # 실제로는 코드 생성 도구 등
            name="backend_dev",
            prompt="백엔드 개발자입니다. 서버 로직을 구현합니다."
        )
        
        frontend_dev = create_react_agent(
            model=self.llm,
            tools=[],
            name="frontend_dev", 
            prompt="프론트엔드 개발자입니다. UI를 구현합니다."
        )
        
        # 엔지니어링팀 감독자
        eng_supervisor = create_supervisor(
            agents=[backend_dev, frontend_dev],
            model=self.llm,
            prompt="""엔지니어링팀 감독자입니다.
            - 백엔드 작업 → backend_dev
            - 프론트엔드 작업 → frontend_dev
            개발 작업을 관리합니다."""
        )
        
        return eng_supervisor.compile(name="engineering_team")
    
    def create_top_supervisor(self):
        """최상위 감독자 생성"""
        
        def top_supervisor_node(state: MessagesState) -> Command:
            """최상위 감독자 로직"""
            
            last_message = state["messages"][-1].content
            
            # 작업 유형 분석
            if any(word in last_message.lower() for word in ["research", "조사", "분석", "데이터"]):
                next_team = "research_team"
            elif any(word in last_message.lower() for word in ["개발", "구현", "코드", "프로그램"]):
                next_team = "engineering_team"
            else:
                # 기본값
                next_team = "research_team"
            
            return Command(
                goto=next_team,
                update={
                    "messages": state["messages"] + [
                        AIMessage(content=f"작업을 {next_team}에 할당합니다.")
                    ]
                }
            )
        
        return top_supervisor_node
    
    def build_hierarchy(self):
        """계층 구조 구축"""
        
        graph = StateGraph(MessagesState)
        
        # 노드 추가
        graph.add_node("top_supervisor", self.create_top_supervisor())
        graph.add_node("research_team", self.research_team)
        graph.add_node("engineering_team", self.engineering_team)
        
        # 엣지 설정
        graph.add_edge(START, "top_supervisor")
        graph.add_edge("research_team", END)
        graph.add_edge("engineering_team", END)
        
        self.system = graph.compile()
    
    def run(self, query: str):
        """시스템 실행"""
        result = self.system.invoke({
            "messages": [HumanMessage(content=query)]
        })
        
        print("\n🏢 계층적 시스템 실행 결과")
        print("=" * 50)
        
        for msg in result["messages"]:
            if isinstance(msg, AIMessage):
                print(f"[{msg.name or 'System'}]: {msg.content}")

# 실행
hierarchical = HierarchicalSystem()
hierarchical.run("최신 AI 트렌드를 조사하고 분석해주세요")
hierarchical.run("간단한 웹 애플리케이션을 개발해주세요")
```

---

## 6. 에이전트 간 통신 패턴

### 6.1 통신 패턴 구현

```python
from langgraph.types import Send
from datetime import datetime

class CommunicationPatterns:
    """다양한 통신 패턴 구현"""
    
    def __init__(self):
        self.llm = ChatOpenAI(model="gpt-4o-mini")
    
    # 패턴 1: 자식 → 부모 복귀
    def child_to_parent_pattern(self):
        """자식이 부모에게 복귀하는 패턴"""
        
        def child_agent(state: MessagesState) -> Command:
            # 작업 수행
            result = "자식 에이전트 작업 완료"
            
            # 부모로 복귀
            return Command(
                goto="parent_supervisor",
                update={"messages": state["messages"] + [AIMessage(content=result)]},
                graph=Command.PARENT  # 중요: 부모 그래프에서 실행
            )
        
        return child_agent
    
    # 패턴 2: 형제 간 핸드오프
    def sibling_handoff_pattern(self):
        """형제 에이전트 간 핸드오프"""
        
        def agent_a(state: MessagesState) -> Command:
            # Agent B로 핸드오프
            return Command(
                goto="agent_b",
                update={
                    "messages": state["messages"] + [
                        AIMessage(content="A에서 B로 작업 전달")
                    ],
                    "handoff_context": {"from": "agent_a", "timestamp": datetime.now()}
                },
                graph=Command.PARENT  # 같은 부모 아래 형제
            )
        
        return agent_a
    
    # 패턴 3: 브로드캐스트
    def broadcast_pattern(self):
        """여러 에이전트에게 동시 전달"""
        
        def coordinator(state: MessagesState) -> Command:
            task = state["messages"][-1].content
            
            # 여러 에이전트에게 동시 전송
            return Command(
                goto=[
                    Send("agent_1", {"task": f"Task 1: {task}"}),
                    Send("agent_2", {"task": f"Task 2: {task}"}),
                    Send("agent_3", {"task": f"Task 3: {task}"})
                ],
                update={"broadcast_sent": True}
            )
        
        return coordinator
    
    # 패턴 4: 조건부 라우팅
    def conditional_routing_pattern(self):
        """조건에 따른 동적 라우팅"""
        
        def smart_router(state: MessagesState) -> Command:
            message = state["messages"][-1].content
            
            # 우선순위 분석
            if "urgent" in message.lower():
                next_node = "high_priority_handler"
                priority = "HIGH"
            elif "simple" in message.lower():
                next_node = "simple_task_handler"
                priority = "LOW"
            else:
                next_node = "normal_handler"
                priority = "NORMAL"
            
            return Command(
                goto=next_node,
                update={
                    "priority": priority,
                    "routed_at": datetime.now().isoformat()
                }
            )
        
        return smart_router
    
    # 패턴 5: 순환 처리
    def circular_pattern(self):
        """에이전트 간 순환 처리"""
        
        class CircularState(MessagesState):
            iteration: int
            max_iterations: int
            agents_cycle: List[str]
            current_index: int
        
        def circular_agent(state: CircularState) -> Command:
            iteration = state.get("iteration", 0)
            max_iterations = state.get("max_iterations", 3)
            agents = state.get("agents_cycle", ["editor", "reviewer", "finalizer"])
            current_idx = state.get("current_index", 0)
            
            # 종료 조건
            if iteration >= max_iterations:
                return Command(goto=END)
            
            # 다음 에이전트
            next_idx = (current_idx + 1) % len(agents)
            next_agent = agents[next_idx]
            
            return Command(
                goto=next_agent,
                update={
                    "iteration": iteration + 1 if next_idx == 0 else iteration,
                    "current_index": next_idx,
                    "messages": state["messages"] + [
                        AIMessage(content=f"Processed by {agents[current_idx]}")
                    ]
                }
            )
        
        return circular_agent

# 통합 예제
class IntegratedCommunicationSystem:
    """모든 통신 패턴을 통합한 시스템"""
    
    def __init__(self):
        self.patterns = CommunicationPatterns()
        self.build_system()
    
    def build_system(self):
        """통합 시스템 구축"""
        
        # 상태 정의
        class ComplexState(MessagesState):
            priority: str
            broadcast_results: dict
            handoff_history: List[dict]
            current_pattern: str
        
        graph = StateGraph(ComplexState)
        
        # 패턴별 노드 추가
        graph.add_node("router", self.patterns.conditional_routing_pattern())
        graph.add_node("broadcaster", self.patterns.broadcast_pattern())
        graph.add_node("child_agent", self.patterns.child_to_parent_pattern())
        graph.add_node("agent_a", self.patterns.sibling_handoff_pattern())
        
        # 핸들러 노드
        graph.add_node("high_priority_handler", self.create_handler("high"))
        graph.add_node("normal_handler", self.create_handler("normal"))
        graph.add_node("simple_task_handler", self.create_handler("simple"))
        
        # 엣지 설정
        graph.add_edge(START, "router")
        
        self.system = graph.compile()
    
    def create_handler(self, priority: str):
        """핸들러 생성"""
        def handler(state):
            return {
                "messages": state["messages"] + [
                    AIMessage(content=f"Handled with {priority} priority")
                ]
            }
        return handler
    
    def run_examples(self):
        """다양한 패턴 실행"""
        
        examples = [
            "urgent: 서버 장애 발생",
            "simple: 날씨 정보 확인",
            "복잡한 데이터 분석 요청"
        ]
        
        for example in examples:
            print(f"\n📨 입력: {example}")
            result = self.system.invoke({
                "messages": [HumanMessage(content=example)]
            })
            print(f"✅ 처리 완료: Priority = {result.get('priority', 'N/A')}")

# 실행
comm_system = IntegratedCommunicationSystem()
comm_system.run_examples()
```

---

## 7. 실전 프로젝트: 뉴스 분석 시스템

### 7.1 완전한 뉴스 분석 Multi-Agent System

```python
import json
from typing import Dict, Any
from datetime import datetime

class NewsAnalysisSystem:
    """뉴스 수집, 분석, 요약을 수행하는 멀티에이전트 시스템"""
    
    def __init__(self):
        self.llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        self.setup_tools()
        self.setup_agents()
        self.build_system()
    
    def setup_tools(self):
        """도구 설정"""
        
        @tool
        def fetch_news(topic: str, count: int = 5) -> str:
            """뉴스 기사 수집
            
            Args:
                topic: 검색할 주제
                count: 가져올 기사 수
            """
            # 실제로는 뉴스 API 사용
            articles = [
                {
                    "title": f"{topic} 관련 뉴스 {i+1}",
                    "content": f"{topic}에 대한 중요한 내용 {i+1}...",
                    "date": datetime.now().isoformat(),
                    "source": f"뉴스소스{i+1}"
                }
                for i in range(count)
            ]
            return json.dumps(articles, ensure_ascii=False)
        
        @tool
        def sentiment_analysis(text: str) -> str:
            """감정 분석
            
            Args:
                text: 분석할 텍스트
            """
            # 간단한 감정 분석 시뮬레이션
            keywords = {
                "positive": ["좋은", "성장", "증가", "개선", "혁신"],
                "negative": ["나쁜", "하락", "감소", "문제", "위기"],
                "neutral": ["유지", "보통", "평균", "일반", "보고"]
            }
            
            sentiment = "neutral"
            score = 0
            
            for word in keywords["positive"]:
                if word in text:
                    score += 1
            for word in keywords["negative"]:
                if word in text:
                    score -= 1
            
            if score > 0:
                sentiment = "positive"
            elif score < 0:
                sentiment = "negative"
            
            return json.dumps({
                "sentiment": sentiment,
                "score": score,
                "confidence": 0.85
            }, ensure_ascii=False)
        
        @tool
        def extract_entities(text: str) -> str:
            """개체명 추출
            
            Args:
                text: 분석할 텍스트
            """
            # 간단한 개체명 추출 시뮬레이션
            entities = {
                "organizations": ["회사A", "기관B"],
                "persons": ["인물1", "인물2"],
                "locations": ["서울", "뉴욕"],
                "dates": [datetime.now().isoformat()]
            }
            return json.dumps(entities, ensure_ascii=False)
        
        @tool
        def generate_summary(articles: str, analysis: str) -> str:
            """요약 생성
            
            Args:
                articles: 기사 목록
                analysis: 분석 결과
            """
            summary = f"""
            📰 뉴스 요약 보고서
            ================
            
            수집된 기사: {len(json.loads(articles))}개
            
            주요 내용:
            - 전반적인 감정: {json.loads(analysis).get('sentiment', 'neutral')}
            - 핵심 키워드: 분석 완료
            - 주요 인물/조직: 식별 완료
            
            상세 분석이 완료되었습니다.
            """
            return summary
        
        self.tools = {
            "fetch_news": fetch_news,
            "sentiment_analysis": sentiment_analysis,
            "extract_entities": extract_entities,
            "generate_summary": generate_summary
        }
    
    def setup_agents(self):
        """에이전트 설정"""
        
        # 1. News Collector
        self.news_collector = create_react_agent(
            model=self.llm,
            tools=[self.tools["fetch_news"]],
            name="news_collector",
            prompt="""당신은 뉴스 수집 전문가입니다.
            주어진 주제에 대한 최신 뉴스를 수집하세요.
            수집한 뉴스를 JSON 형식으로 정리하세요."""
        )
        
        # 2. Sentiment Analyzer
        self.sentiment_analyzer = create_react_agent(
            model=self.llm,
            tools=[self.tools["sentiment_analysis"]],
            name="sentiment_analyzer",
            prompt="""당신은 감정 분석 전문가입니다.
            뉴스 기사의 감정을 분석하고 긍정/부정/중립으로 분류하세요.
            전체적인 톤과 감정 점수를 제공하세요."""
        )
        
        # 3. Entity Extractor
        self.entity_extractor = create_react_agent(
            model=self.llm,
            tools=[self.tools["extract_entities"]],
            name="entity_extractor",
            prompt="""당신은 개체명 인식 전문가입니다.
            텍스트에서 인물, 조직, 장소, 날짜 등을 추출하세요.
            추출된 개체를 카테고리별로 정리하세요."""
        )
        
        # 4. Report Generator
        self.report_generator = create_react_agent(
            model=self.llm,
            tools=[self.tools["generate_summary"]],
            name="report_generator",
            prompt="""당신은 보고서 작성 전문가입니다.
            수집된 뉴스와 분석 결과를 바탕으로 종합 보고서를 작성하세요.
            핵심 인사이트와 추천사항을 포함하세요."""
        )
        
        # 5. Supervisor
        self.supervisor = create_supervisor(
            agents=[
                self.news_collector,
                self.sentiment_analyzer,
                self.entity_extractor,
                self.report_generator
            ],
            model=self.llm,
            prompt="""당신은 뉴스 분석팀 관리자입니다.
            
            작업 프로세스:
            1. news_collector: 뉴스 수집
            2. sentiment_analyzer: 감정 분석
            3. entity_extractor: 개체 추출
            4. report_generator: 보고서 생성
            
            각 단계를 순차적으로 진행하고 결과를 통합하세요."""
        )
    
    def build_system(self):
        """시스템 구축"""
        
        # 커스텀 상태
        class NewsAnalysisState(MessagesState):
            topic: str
            collected_news: List[dict]
            sentiment_results: dict
            entities: dict
            final_report: str
            workflow_stage: str
        
        # 워크플로우 관리자
        def workflow_manager(state: NewsAnalysisState) -> Command:
            """워크플로우 단계 관리"""
            
            stage = state.get("workflow_stage", "start")
            
            stages_map = {
                "start": "news_collector",
                "collected": "sentiment_analyzer",
                "analyzed": "entity_extractor",
                "extracted": "report_generator",
                "completed": END
            }
            
            next_stage = stages_map.get(stage, END)
            
            return Command(
                goto=next_stage,
                update={"workflow_stage": next_stage}
            )
        
        # 그래프 구성
        graph = StateGraph(NewsAnalysisState)
        
        # 노드 추가
        graph.add_node("workflow_manager", workflow_manager)
        graph.add_node("supervisor", self.supervisor)
        
        # 엣지 설정
        graph.add_edge(START, "supervisor")
        
        self.system = graph.compile()
    
    def analyze_news(self, topic: str):
        """뉴스 분석 실행"""
        
        print(f"\n📊 뉴스 분석 시작: {topic}")
        print("=" * 60)
        
        result = self.system.invoke({
            "messages": [HumanMessage(
                content=f"다음 주제에 대한 뉴스를 수집하고 분석해주세요: {topic}"
            )],
            "topic": topic,
            "workflow_stage": "start"
        })
        
        # 결과 출력
        print("\n📈 분석 결과")
        print("-" * 60)
        
        for msg in result["messages"][-3:]:  # 마지막 3개 메시지
            if isinstance(msg, AIMessage):
                print(f"\n[{msg.name or 'Agent'}]:")
                print(msg.content[:500] + "..." if len(msg.content) > 500 else msg.content)
        
        return result

# 메인 실행 함수
def main():
    """메인 실행 함수"""
    
    print("🚀 LangGraph Multi-Agent 뉴스 분석 시스템")
    print("=" * 60)
    
    # 시스템 초기화
    news_system = NewsAnalysisSystem()
    
    # 분석할 주제들
    topics = [
        "인공지능 최신 동향",
        "기후변화 대응 정책",
        "글로벌 경제 전망"
    ]
    
    # 각 주제 분석
    for topic in topics:
        try:
            result = news_system.analyze_news(topic)
            
            # 결과 저장 (선택적)
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"news_analysis_{timestamp}_{topic[:10]}.json"
            
            with open(filename, "w", encoding="utf-8") as f:
                json.dump({
                    "topic": topic,
                    "timestamp": timestamp,
                    "messages": [msg.content for msg in result["messages"] if isinstance(msg, AIMessage)]
                }, f, ensure_ascii=False, indent=2)
            
            print(f"\n✅ 분석 결과가 {filename}에 저장되었습니다.")
            
        except Exception as e:
            print(f"\n❌ 오류 발생: {e}")
        
        print("\n" + "=" * 60)

if __name__ == "__main__":
    main()
```

### 7.2 실행 및 모니터링

```python
class SystemMonitor:
    """시스템 모니터링 및 디버깅"""
    
    def __init__(self, system):
        self.system = system
        self.execution_history = []
    
    def trace_execution(self, input_data: dict):
        """실행 추적"""
        
        start_time = datetime.now()
        
        # 스트리밍으로 실행 추적
        trace_data = {
            "input": input_data,
            "steps": [],
            "start_time": start_time
        }
        
        for step in self.system.stream(input_data):
            step_info = {
                "timestamp": datetime.now().isoformat(),
                "node": list(step.keys())[0] if step else "unknown",
                "output": str(step)[:200]  # 처음 200자만
            }
            trace_data["steps"].append(step_info)
            
            # 실시간 출력
            print(f"⚙️ [{step_info['timestamp']}] {step_info['node']}")
        
        trace_data["end_time"] = datetime.now()
        trace_data["duration"] = (trace_data["end_time"] - start_time).total_seconds()
        
        self.execution_history.append(trace_data)
        
        return trace_data
    
    def generate_report(self):
        """실행 보고서 생성"""
        
        print("\n📊 시스템 실행 보고서")
        print("=" * 60)
        
        for i, execution in enumerate(self.execution_history):
            print(f"\n실행 #{i+1}")
            print(f"  시작: {execution['start_time']}")
            print(f"  소요 시간: {execution['duration']:.2f}초")
            print(f"  처리 단계: {len(execution['steps'])}개")
            
            # 단계별 시간 분석
            if execution['steps']:
                print("\n  단계별 실행:")
                for step in execution['steps'][:5]:  # 처음 5단계만
                    print(f"    - {step['node']}: {step['timestamp']}")

# 모니터링과 함께 실행
if __name__ == "__main__":
    # 시스템 생성
    news_system = NewsAnalysisSystem()
    
    # 모니터 연결
    monitor = SystemMonitor(news_system.system)
    
    # 모니터링과 함께 실행
    input_data = {
        "messages": [HumanMessage(content="AI 스타트업 투자 동향을 분석해주세요")]
    }
    
    trace = monitor.trace_execution(input_data)
    
    # 보고서 생성
    monitor.generate_report()
```

---

## 8. 모범 사례 및 팁

### 8.1 에이전트 설계 원칙

```python
"""
1. 단일 책임 원칙 (Single Responsibility)
   - 각 에이전트는 하나의 명확한 역할만 담당
   
2. 명확한 인터페이스
   - 입력/출력 형식을 명확히 정의
   - 상태 스키마를 일관성 있게 유지
   
3. 오류 처리
   - 각 에이전트에 try-except 구현
   - 실패 시 graceful degradation
   
4. 성능 최적화
   - 불필요한 에이전트 호출 최소화
   - 병렬 처리 가능한 작업은 브로드캐스트 활용
   
5. 디버깅 용이성
   - 각 단계마다 로깅
   - 상태 변화 추적
"""

class BestPracticeAgent:
    """모범 사례를 적용한 에이전트"""
    
    def __init__(self, name: str, role: str):
        self.name = name
        self.role = role
        self.logger = self.setup_logger()
    
    def setup_logger(self):
        """로깅 설정"""
        import logging
        
        logger = logging.getLogger(self.name)
        logger.setLevel(logging.INFO)
        
        handler = logging.StreamHandler()
        formatter = logging.Formatter(
            f'[%(asctime)s] [{self.name}] %(levelname)s: %(message)s'
        )
        handler.setFormatter(formatter)
        logger.addHandler(handler)
        
        return logger
    
    def create_node(self):
        """에이전트 노드 생성"""
        
        def agent_node(state: MessagesState) -> Command:
            try:
                # 시작 로깅
                self.logger.info(f"Processing started for {self.role}")
                
                # 입력 검증
                if not state.get("messages"):
                    raise ValueError("No messages in state")
                
                # 작업 수행
                result = self.process_task(state)
                
                # 성공 로깅
                self.logger.info("Processing completed successfully")
                
                return Command(
                    goto=self.determine_next_node(result),
                    update=self.create_update(state, result)
                )
                
            except Exception as e:
                # 오류 로깅
                self.logger.error(f"Error occurred: {e}")
                
                # 오류 처리
                return Command(
                    goto="error_handler",
                    update={
                        "error": str(e),
                        "failed_agent": self.name
                    }
                )
        
        return agent_node
    
    def process_task(self, state: MessagesState) -> dict:
        """실제 작업 처리 (override 필요)"""
        raise NotImplementedError
    
    def determine_next_node(self, result: dict) -> str:
        """다음 노드 결정 (override 필요)"""
        raise NotImplementedError
    
    def create_update(self, state: MessagesState, result: dict) -> dict:
        """상태 업데이트 생성"""
        return {
            "messages": state["messages"] + [
                AIMessage(content=result.get("content", ""), name=self.name)
            ],
            "last_agent": self.name,
            "timestamp": datetime.now().isoformat()
        }
```

### 8.2 테스트 전략

```python
import unittest
from unittest.mock import Mock, patch

class TestMultiAgentSystem(unittest.TestCase):
    """멀티에이전트 시스템 테스트"""
    
    def setUp(self):
        """테스트 설정"""
        self.system = NewsAnalysisSystem()
    
    def test_agent_creation(self):
        """에이전트 생성 테스트"""
        self.assertIsNotNone(self.system.news_collector)
        self.assertIsNotNone(self.system.sentiment_analyzer)
    
    def test_workflow_execution(self):
        """워크플로우 실행 테스트"""
        test_input = {
            "messages": [HumanMessage(content="테스트 주제")]
        }
        
        result = self.system.system.invoke(test_input)
        
        self.assertIn("messages", result)
        self.assertTrue(len(result["messages"]) > 1)
    
    def test_error_handling(self):
        """오류 처리 테스트"""
        with patch.object(self.system, 'analyze_news', side_effect=Exception("Test error")):
            try:
                self.system.analyze_news("오류 테스트")
            except Exception as e:
                self.assertEqual(str(e), "Test error")
    
    def test_communication_patterns(self):
        """통신 패턴 테스트"""
        patterns = CommunicationPatterns()
        
        # 각 패턴 테스트
        child_to_parent = patterns.child_to_parent_pattern()
        self.assertIsNotNone(child_to_parent)
        
        broadcast = patterns.broadcast_pattern()
        self.assertIsNotNone(broadcast)

# 테스트 실행
if __name__ == "__main__":
    unittest.main()
```

---

## 9. 결론 및 추가 리소스

### 9.1 핵심 요약

```python
"""
LangGraph Multi-Agent 시스템 핵심 포인트:

1. **아키텍처 패턴**
   - Supervisor: 중앙 집중식 제어
   - Swarm: 분산형 자율 협업
   - Hierarchical: 다층 구조

2. **통신 메커니즘**
   - Command 타입으로 유연한 제어 흐름
   - graph=Command.PARENT로 부모 그래프 접근
   - Send로 병렬 처리

3. **상태 관리**
   - MessagesState 기본 상태
   - 커스텀 상태로 확장 가능
   - Checkpointer로 상태 지속성

4. **도구 통합**
   - @tool 데코레이터로 도구 정의
   - create_react_agent로 도구 사용 에이전트 생성

5. **모니터링 및 디버깅**
   - 스트리밍으로 실시간 추적
   - 로깅으로 문제 진단
"""
```

### 9.2 추가 리소스

- [LangGraph 공식 문서](https://langchain-ai.github.io/langgraph/)
- [LangGraph GitHub](https://github.com/langchain-ai/langgraph)
- [LangChain Blog](https://blog.langchain.com/)
- [Community Discord](https://discord.gg/langchain)

### 9.3 다음 단계

1. **고급 기능 탐색**
   - Persistent checkpointing
   - Streaming과 실시간 처리
   - Human-in-the-loop 패턴

2. **프로덕션 고려사항**
   - 확장성 (Scale)
   - 모니터링 (Observability)
   - 보안 (Security)

3. **실제 적용 사례**
   - 고객 서비스 봇
   - 연구 어시스턴트
   - 콘텐츠 생성 파이프라인
   - 데이터 분석 시스템

---

**작성자**: LangGraph Tutorial  
**버전**: 1.0.0  
**최종 수정**: 2025년 1월  
**라이선스**: MIT

In [ ]:
# LangGraph 멀티에이전트 통신 패턴 상세 가이드

## 1. 그래프 계층 구조 이해

### 기본 개념
```python
# LangGraph의 그래프 계층
# 
# [최상위 그래프 (Top-level Graph)]
#         ↓
# [부모 그래프 (Parent Graph)]
#     ↓        ↓
# [자식1]    [자식2]  ← 자식 그래프/에이전트 (Child Graph/Agent)
```

Command 타입을 사용하면 graph 파라미터를 통해 다음 노드가 실행될 그래프 범위를 지정할 수 있습니다. Command.PARENT를 사용하면 부모 그래프의 노드로 이동할 수 있습니다.

---

## 2. 통신 케이스별 상세 구현

### **케이스 1: 자식 → 부모 그래프로 복귀**

가장 일반적인 패턴으로, 작업을 완료한 자식 에이전트가 부모(감독자)에게 제어권을 반환합니다.

```python
from langgraph.types import Command
from langgraph.graph import MessagesState
from typing import Literal

def child_agent_node(state: MessagesState) -> Command[Literal["supervisor"]]:
    """자식 에이전트가 작업 완료 후 부모로 복귀"""
    
    # 작업 수행
    result = perform_task(state)
    
    # 부모 그래프의 supervisor 노드로 복귀
    return Command(
        goto="supervisor",  # 부모 그래프의 노드 이름
        update={"messages": state["messages"] + [result]},
        graph=Command.PARENT  # 핵심: 부모 그래프 범위에서 실행
    )

# 실제 구현 예제
def create_return_to_supervisor_tool():
    @tool
    def return_to_supervisor(
        task_result: str,
        state: Annotated[MessagesState, InjectedState],
        tool_call_id: Annotated[str, InjectedToolCallId]
    ) -> Command:
        tool_message = ToolMessage(
            content=f"작업 완료: {task_result}",
            tool_call_id=tool_call_id
        )
        
        return Command(
            goto="supervisor",
            update={"messages": state["messages"] + [tool_message]},
            graph=Command.PARENT
        )
    return return_to_supervisor
```

---

### **케이스 2: 부모 → 자식 에이전트 호출**

감독자가 특정 자식 에이전트에게 작업을 할당하는 패턴입니다.

```python
from langgraph.types import Send

def supervisor_node(state: MessagesState) -> Command:
    """감독자가 자식 에이전트를 호출"""
    
    # LLM으로 다음 에이전트 결정
    decision = llm.invoke(state["messages"])
    
    # 옵션 1: 단순 이동 (같은 그래프 레벨)
    if decision.next_agent in ["agent_a", "agent_b"]:
        return Command(
            goto=decision.next_agent,
            update={"messages": state["messages"]}
        )
    
    # 옵션 2: Send를 사용한 명시적 데이터 전달
    task_data = prepare_task_data(state)
    return Command(
        goto=[Send(decision.next_agent, task_data)],
        update={"task_assigned": True}
    )

# 자식 에이전트를 서브그래프로 호출
def call_subgraph_team(state: MessagesState) -> Command[Literal["supervisor"]]:
    """서브그래프(팀)를 호출하고 결과 받기"""
    
    # 서브그래프 실행
    subgraph_result = research_team_graph.invoke({
        "messages": state["messages"],
        "task": state.get("current_task")
    })
    
    # 결과를 부모 상태에 통합
    return Command(
        goto="supervisor",
        update={
            "messages": state["messages"] + [
                HumanMessage(
                    content=subgraph_result["final_answer"],
                    name="research_team"
                )
            ],
            "team_results": subgraph_result
        }
    )
```

---

### **케이스 3: 형제 에이전트 간 직접 핸드오프**

같은 레벨의 에이전트끼리 직접 제어권을 넘기는 패턴입니다.

```python
def create_peer_handoff_tool(target_agent: str):
    """동일 레벨 에이전트로 핸드오프"""
    
    @tool(f"handoff_to_{target_agent}")
    def peer_handoff(
        context: str,
        state: Annotated[MessagesState, InjectedState],
        tool_call_id: Annotated[str, InjectedToolCallId]
    ) -> Command:
        
        # 현재 에이전트의 컨텍스트를 다음 에이전트에게 전달
        handoff_message = {
            "role": "assistant",
            "content": f"핸드오프 from {state.get('current_agent')}: {context}",
            "metadata": {
                "from_agent": state.get('current_agent'),
                "to_agent": target_agent,
                "handoff_reason": context
            }
        }
        
        return Command(
            goto=target_agent,
            update={
                "messages": state["messages"] + [handoff_message],
                "current_agent": target_agent,
                "handoff_history": state.get("handoff_history", []) + [
                    {"from": state.get('current_agent'), "to": target_agent}
                ]
            },
            graph=Command.PARENT  # 부모 그래프 레벨에서 실행
        )
    
    return peer_handoff

# 실제 사용 예제
research_agent = create_react_agent(
    model=llm,
    tools=[
        search_tool,
        create_peer_handoff_tool("math_agent"),  # math_agent로 핸드오프
        create_peer_handoff_tool("writer_agent")  # writer_agent로 핸드오프
    ],
    prompt="연구를 수행하고 필요시 다른 전문가에게 넘기세요."
)
```

---

### **케이스 4: 계층적 통신 (다단계 그래프)**

여러 계층의 그래프가 있을 때의 통신 패턴입니다.

```python
# 3단계 계층 구조 예제
class HierarchicalState(MessagesState):
    current_level: int
    team_assignments: dict
    aggregated_results: list

def top_level_supervisor(state: HierarchicalState) -> Command:
    """최상위 감독자"""
    
    # 팀 레벨 감독자들에게 작업 분배
    assignments = analyze_and_distribute_tasks(state["messages"])
    
    return Command(
        goto=assignments[0]["team"],  # 첫 번째 팀으로
        update={
            "team_assignments": assignments,
            "current_level": 1
        }
    )

def team_supervisor(state: HierarchicalState) -> Command:
    """중간 레벨 팀 감독자"""
    
    team_name = state.get("current_team")
    task = state["team_assignments"][team_name]
    
    # 개별 에이전트에게 세부 작업 할당
    if needs_specialist(task):
        return Command(
            goto=f"{team_name}_specialist",
            update={
                "current_level": 2,
                "current_task": task
            }
        )
    
    # 작업 완료 시 상위로 복귀
    return Command(
        goto="top_supervisor",
        update={
            "aggregated_results": state["aggregated_results"] + [result],
            "current_level": 0
        },
        graph=Command.PARENT
    )

def specialist_agent(state: HierarchicalState) -> Command:
    """최하위 전문가 에이전트"""
    
    result = execute_specialized_task(state["current_task"])
    
    # 두 단계 위로 복귀 (specialist → team → top)
    return Command(
        goto="team_supervisor",
        update={
            "messages": state["messages"] + [result],
            "current_level": 1
        },
        graph=Command.PARENT
    )

# 계층적 그래프 구성
top_graph = StateGraph(HierarchicalState)
top_graph.add_node("top_supervisor", top_level_supervisor)

# 팀 레벨 서브그래프
research_team = StateGraph(HierarchicalState)
research_team.add_node("team_supervisor", team_supervisor)
research_team.add_node("research_specialist", specialist_agent)

calc_team = StateGraph(HierarchicalState)
calc_team.add_node("team_supervisor", team_supervisor)
calc_team.add_node("calc_specialist", specialist_agent)

# 컴파일
top_graph.add_node("research_team", research_team.compile())
top_graph.add_node("calc_team", calc_team.compile())
```

---

### **케이스 5: 조건부 분기 통신**

조건에 따라 다른 경로로 통신하는 패턴입니다.

```python
def conditional_router(state: MessagesState) -> Command:
    """조건에 따른 동적 라우팅"""
    
    message = state["messages"][-1]
    
    # 긴급도에 따른 분기
    if "urgent" in message.content.lower():
        # 직접 상위 감독자에게 에스컬레이션
        return Command(
            goto="emergency_handler",
            update={
                "priority": "high",
                "escalation_reason": "urgent keyword detected"
            },
            graph=Command.PARENT
        )
    
    # 복잡도에 따른 분기
    complexity = assess_complexity(message)
    
    if complexity > 0.8:
        # 전문가 팀으로 전달
        return Command(
            goto=[
                Send("expert_team", {
                    "messages": state["messages"],
                    "complexity_score": complexity
                })
            ]
        )
    elif complexity > 0.5:
        # 일반 에이전트로 전달
        return Command(goto="standard_agent")
    else:
        # 간단한 작업은 직접 처리
        response = handle_simple_task(message)
        return Command(
            goto=END,
            update={"messages": state["messages"] + [response]}
        )
```

---

### **케이스 6: 브로드캐스트 통신**

하나의 에이전트가 여러 에이전트에게 동시에 작업을 전달하는 패턴입니다.

```python
def broadcast_coordinator(state: MessagesState) -> Command:
    """여러 에이전트에게 동시 작업 할당"""
    
    task = state["messages"][-1].content
    
    # 병렬 처리를 위한 다중 Send
    return Command(
        goto=[
            Send("researcher", {
                "messages": [{"role": "user", "content": f"Research: {task}"}],
                "task_id": "research_001"
            }),
            Send("analyzer", {
                "messages": [{"role": "user", "content": f"Analyze: {task}"}],
                "task_id": "analysis_001"
            }),
            Send("validator", {
                "messages": [{"role": "user", "content": f"Validate: {task}"}],
                "task_id": "validation_001"
            })
        ],
        update={"broadcast_initiated": True}
    )

# 결과 수집기
def result_aggregator(state: MessagesState) -> Command:
    """병렬 처리 결과 수집"""
    
    results = state.get("parallel_results", {})
    
    # 모든 결과가 도착했는지 확인
    if len(results) >= 3:  # 3개 에이전트의 결과
        final_result = synthesize_results(results)
        
        return Command(
            goto="supervisor",
            update={
                "messages": state["messages"] + [final_result],
                "final_synthesis": final_result
            },
            graph=Command.PARENT
        )
    
    # 아직 대기 중
    return Command(
        goto="wait",
        update={"waiting_for_results": True}
    )
```

---

### **케이스 7: 순환 핸드오프 (Round-Robin)**

에이전트들이 순환하며 작업을 처리하는 패턴입니다.

```python
class RoundRobinState(MessagesState):
    agent_queue: list
    current_agent_index: int
    iteration_count: int
    max_iterations: int

def round_robin_agent(state: RoundRobinState) -> Command:
    """순환 방식으로 다음 에이전트 선택"""
    
    current_idx = state.get("current_agent_index", 0)
    agents = state.get("agent_queue", [])
    iteration = state.get("iteration_count", 0)
    
    # 작업 수행
    result = process_with_current_agent(state, agents[current_idx])
    
    # 종료 조건 확인
    if is_task_complete(result) or iteration >= state["max_iterations"]:
        return Command(
            goto=END,
            update={"messages": state["messages"] + [result]}
        )
    
    # 다음 에이전트 선택
    next_idx = (current_idx + 1) % len(agents)
    next_agent = agents[next_idx]
    
    return Command(
        goto=next_agent,
        update={
            "messages": state["messages"] + [result],
            "current_agent_index": next_idx,
            "iteration_count": iteration + 1
        }
    )

# 순환 그래프 설정
round_robin_graph = StateGraph(RoundRobinState)
agents = ["editor", "reviewer", "finalizer"]

for agent in agents:
    round_robin_graph.add_node(agent, round_robin_agent)

# 순환 엣지 추가
for i, agent in enumerate(agents):
    next_agent = agents[(i + 1) % len(agents)]
    round_robin_graph.add_edge(agent, next_agent)
```

---

### **케이스 8: 컨텍스트 보존 핸드오프**

핸드오프 시 특정 컨텍스트만 선택적으로 전달하는 패턴입니다.

```python
class ContextualState(MessagesState):
    shared_context: dict  # 모든 에이전트가 공유
    private_context: dict  # 에이전트별 private 데이터
    
def create_contextual_handoff(target: str, context_keys: list):
    """선택적 컨텍스트 전달"""
    
    @tool(f"handoff_to_{target}_with_context")
    def contextual_handoff(
        state: Annotated[ContextualState, InjectedState],
        additional_info: str = ""
    ) -> Command:
        
        # 필요한 컨텍스트만 추출
        filtered_context = {
            k: state["shared_context"][k] 
            for k in context_keys 
            if k in state["shared_context"]
        }
        
        # Private 컨텍스트 준비
        private_data = {
            "from_agent": state.get("current_agent"),
            "handoff_time": datetime.now().isoformat(),
            "additional_info": additional_info
        }
        
        return Command(
            goto=target,
            update={
                "shared_context": filtered_context,
                "private_context": {
                    **state.get("private_context", {}),
                    target: private_data
                },
                "messages": state["messages"] + [
                    {
                        "role": "system",
                        "content": f"Context transferred: {', '.join(context_keys)}"
                    }
                ]
            },
            graph=Command.PARENT
        )
    
    return contextual_handoff

# 사용 예제
research_agent = create_react_agent(
    model=llm,
    tools=[
        search_tool,
        # 분석가에게는 raw_data와 sources만 전달
        create_contextual_handoff("analyst", ["raw_data", "sources"]),
        # 작성자에게는 summary와 key_points만 전달  
        create_contextual_handoff("writer", ["summary", "key_points"])
    ]
)
```

---

## 3. 통신 패턴 선택 가이드

### 패턴 선택 기준

| 시나리오 | 추천 패턴 | Command 설정 |
|---------|---------|-------------|
| 작업 완료 후 보고 | 자식→부모 복귀 | `graph=Command.PARENT` |
| 전문가 에이전트 호출 | 부모→자식 호출 | `goto="agent_name"` |
| 동료 간 협업 | 형제 간 핸드오프 | `graph=Command.PARENT` |
| 복잡한 작업 분해 | 계층적 통신 | 다단계 `Command.PARENT` |
| 병렬 처리 | 브로드캐스트 | 다중 `Send()` |
| 반복 개선 | 순환 핸드오프 | 조건부 `goto` |

---

## 4. 실전 예제: 통합 멀티에이전트 시스템

```python
# 모든 통신 패턴을 활용한 통합 시스템
class IntegratedMultiAgentSystem:
    def __init__(self):
        self.setup_agents()
        self.setup_communication_tools()
        self.build_graph()
    
    def setup_communication_tools(self):
        """다양한 통신 도구 설정"""
        
        # 1. 부모 복귀 도구
        self.return_to_supervisor = Command(
            goto="main_supervisor",
            graph=Command.PARENT
        )
        
        # 2. 형제 간 핸드오프
        self.peer_handoffs = {
            agent: create_peer_handoff_tool(agent)
            for agent in ["researcher", "analyst", "writer"]
        }
        
        # 3. 조건부 에스컬레이션
        self.escalation = lambda severity: Command(
            goto="emergency_handler" if severity > 0.8 else "supervisor",
            graph=Command.PARENT
        )
        
        # 4. 브로드캐스트
        self.broadcast = lambda agents, task: Command(
            goto=[Send(a, {"task": task}) for a in agents]
        )
    
    def build_graph(self):
        """통합 그래프 구성"""
        
        graph = StateGraph(IntegratedState)
        
        # 노드 추가
        graph.add_node("main_supervisor", self.main_supervisor_logic)
        graph.add_node("research_team", self.research_team_graph)
        graph.add_node("analysis_team", self.analysis_team_graph)
        graph.add_node("emergency_handler", self.emergency_handler)
        
        # 조건부 엣지
        graph.add_conditional_edges(
            "main_supervisor",
            self.route_decision,
            {
                "research": "research_team",
                "analysis": "analysis_team",
                "emergency": "emergency_handler",
                "complete": END
            }
        )
        
        return graph.compile()
```

이러한 다양한 통신 패턴을 적절히 조합하면 복잡한 멀티에이전트 시스템을 효과적으로 구현할 수 있습니다.

In [ ]:
네, LangGraph의 최신 문서와 코드를 확인하여 멀티 에이전트 교재를 보완해드리겠습니다.교재를 최신 LangGraph 문서와 코드를 기반으로 보완한 개선된 버전을 제공해드리겠습니다.

# LangGraph 활용 - Multi-Agent 아키텍처 (개정판)

## 환경 설정 및 준비

### (1) Env 환경변수
```python
from dotenv import load_dotenv
load_dotenv()
```

### (2) 기본 라이브러리
```python
import os
from glob import glob
from pprint import pprint
import json
import warnings
warnings.filterwarnings("ignore")
```

### (3) Langsmith tracing 설정
```python
# Langsmith tracing 여부 확인
import os
print(os.getenv('LANGSMITH_TRACING'))
```

### (4) 필수 패키지 설치
```bash
pip install -U langgraph langgraph-supervisor langgraph-swarm langchain-tavily langchain-openai
```

---

## **LangGraph 멀티에이전트 개요**

### **1. 에이전트란?**
- **에이전트**는 LLM을 사용하여 애플리케이션의 제어 흐름을 결정하는 시스템
- LangGraph에서는 각 에이전트가 그래프의 노드로 표현되며, 엣지를 통해 상호 통신

### **2. 멀티 에이전트 시스템이 필요한 이유**
단일 에이전트 시스템의 한계:
- **도구 과부하**: 너무 많은 도구로 인한 잘못된 의사결정
- **컨텍스트 복잡성**: 단일 에이전트가 추적하기 어려운 복잡한 상태
- **전문화 필요**: 플래너, 연구자, 수학 전문가 등 여러 전문 영역 필요

### **3. 멀티 에이전트 시스템의 주요 장점**
- **모듈성**: 개별 에이전트로 분리하여 개발, 테스트, 유지보수 용이
- **전문화**: 특정 도메인에 특화된 에이전트로 전체 성능 향상
- **제어성**: 에이전트 간 통신을 명시적으로 제어 가능

### **4. LangGraph Command 타입 (2024년 12월 출시)**
Command는 노드에서 반환될 때 상태 업데이트뿐만 아니라 다음에 실행할 노드를 지정하는 특별한 타입입니다. 이를 통해 더 유연한 멀티에이전트 아키텍처 구현이 가능합니다.

```python
from langgraph.types import Command
from typing import Literal

def agent(state: MessagesState) -> Command[Literal["agent_a", "agent_b", END]]:
    # LLM 호출 및 다음 에이전트 결정
    return Command(
        goto="agent_b",  # 다음 노드
        update={"messages": [response]},  # 상태 업데이트
        graph=Command.PARENT  # 부모 그래프에서 실행
    )
```

---

## **1. Supervisor 패턴**

Supervisor는 중앙 감독자 에이전트가 개별 에이전트들을 조정하는 멀티에이전트 아키텍처입니다. 감독자는 모든 통신 흐름과 작업 위임을 제어하며, 현재 컨텍스트와 작업 요구사항에 따라 어떤 에이전트를 호출할지 결정합니다.

![Supervisor 패턴](https://langchain-ai.github.io/langgraph/agents/assets/supervisor.png)

### (1) langgraph-supervisor 패키지 사용 (권장)

```python
from langgraph.prebuilt import create_react_agent
from langchain_tavily import TavilySearch
from langchain_core.tools import tool
from IPython.display import Image, display

# 1. 도구 정의
tavily_search = TavilySearch(max_results=3)

@tool
def add(a: float, b: float) -> float:
    """두 수를 더합니다."""
    return a + b

@tool  
def multiply(a: float, b: float) -> float:
    """두 수를 곱합니다."""
    return a * b

@tool
def divide(a: float, b: float) -> float:
    """두 수를 나눕니다."""
    if b == 0:
        return "0으로 나눌 수 없습니다."
    return a / b

# 2. 작업자 에이전트들 생성
research_agent = create_react_agent(
    model="openai:gpt-4o-mini",
    tools=[tavily_search],
    prompt="당신은 연구 전문가입니다. 웹 검색으로 정보를 찾아주세요.",
    name="research_agent"
)

math_agent = create_react_agent(
    model="openai:gpt-4o-mini",
    tools=[add, multiply, divide],
    prompt="당신은 수학 전문가입니다. 계산을 도와드립니다.",
    name="math_agent"
)

# 3. 감독자 시스템 생성
from langgraph_supervisor import create_supervisor
from langchain.chat_models import init_chat_model

supervisor = create_supervisor(
    agents=[research_agent, math_agent],
    model=init_chat_model("openai:gpt-4o"),
    prompt=(
        "당신은 두 명의 에이전트를 관리하는 감독자입니다:\n"
        "- research_agent: 정보 검색 작업을 전담\n"
        "- math_agent: 수학 계산 작업을 전담\n\n"
        "전문성을 고려하여 적절한 에이전트에게 작업을 할당하세요.\n"
        "한 번에 하나의 에이전트에만 작업을 할당하세요."
    ),
    add_handoff_back_messages=True,  # 핸드오프 메시지 자동 추가
    output_mode="full_history"  # 전체 대화 기록 유지
).compile()

# 그래프 시각화
display(Image(supervisor.get_graph().draw_mermaid_png()))
```

### (2) Command를 활용한 커스텀 핸드오프 도구

Command 타입을 사용하면 핸드오프를 쉽게 수행할 수 있습니다. 그래프의 다른 노드로 점프하도록 지정할 수 있으며, 부모 그래프의 노드로도 점프할 수 있습니다.

```python
from typing import Annotated
from langchain_core.tools import tool, InjectedToolCallId
from langchain_core.messages import ToolMessage
from langgraph.prebuilt import InjectedState
from langgraph.graph import MessagesState, StateGraph, START, END
from langgraph.types import Command

def create_handoff_tool(*, agent_name: str, description: str = None):
    """Command를 사용한 개선된 핸드오프 도구"""
    name = f"transfer_to_{agent_name}"
    description = description or f"Transfer to {agent_name}"

    @tool(name, description=description)
    def handoff_tool(
        state: Annotated[MessagesState, InjectedState], 
        tool_call_id: Annotated[str, InjectedToolCallId],
    ) -> Command:
        tool_message = ToolMessage(
            content=f"Successfully transferred to {agent_name}",
            name=name,
            tool_call_id=tool_call_id,
        )
        
        return Command(  
            goto=agent_name,  # 다음 에이전트로 이동
            update={"messages": state["messages"] + [tool_message]},
            graph=Command.PARENT,  # 부모 그래프 범위에서 실행
        )
    
    return handoff_tool

# 핸드오프 도구 생성
transfer_to_research = create_handoff_tool(
    agent_name="research_agent",
    description="정보 검색이 필요할 때 연구 에이전트로 전달"
)

transfer_to_math = create_handoff_tool(
    agent_name="math_agent", 
    description="수학 계산이 필요할 때 수학 에이전트로 전달"
)

transfer_to_supervisor = create_handoff_tool(
    agent_name="supervisor",
    description="작업 완료 후 슈퍼바이저에게 보고"
)

# 에이전트 생성
supervisor_agent = create_react_agent(
    model="openai:gpt-4o",
    tools=[transfer_to_research, transfer_to_math],
    prompt="""당신은 팀 슈퍼바이저입니다. 
    
🔍 연구 작업 → research_agent
🧮 수학 작업 → math_agent

작업을 분석하고 적절한 에이전트에게 전달하세요.""",
    name="supervisor"
)

research_agent = create_react_agent(
    model="openai:gpt-4o-mini",
    tools=[tavily_search, transfer_to_supervisor],
    prompt="연구 전문가입니다. 작업 완료 후 슈퍼바이저에게 보고하세요.",
    name="research_agent"
)

math_agent = create_react_agent(
    model="openai:gpt-4o-mini",
    tools=[add, multiply, divide, transfer_to_supervisor],
    prompt="수학 전문가입니다. 작업 완료 후 슈퍼바이저에게 보고하세요.",
    name="math_agent"
)

# 그래프 구성
supervisor_graph = (
    StateGraph(MessagesState)
    .add_node("supervisor", supervisor_agent)
    .add_node("research_agent", research_agent)
    .add_node("math_agent", math_agent)
    .add_edge(START, "supervisor")
    .compile()
)
```

---

## **2. Swarm 패턴**

Swarm 패턴에서는 에이전트들이 전문성에 따라 서로 제어권을 동적으로 넘겨줍니다. 시스템은 마지막으로 활성화된 에이전트를 기억하여, 후속 상호작용 시 해당 에이전트와 대화를 재개합니다.

![Swarm 패턴](https://langchain-ai.github.io/langgraph/agents/assets/swarm.png)

```python
from langgraph_swarm import create_swarm, create_handoff_tool
from langgraph.checkpoint.memory import InMemorySaver

# 핸드오프 도구 생성
transfer_to_math = create_handoff_tool(
    agent_name="math_agent",
    description="수학 계산이 필요할 때 수학 전문가에게 전달"
)

transfer_to_research = create_handoff_tool(
    agent_name="research_agent", 
    description="정보 검색이 필요할 때 연구 전문가에게 전달"
)

# 에이전트 생성 (상호 핸드오프 가능)
research_agent = create_react_agent(
    model="openai:gpt-4o-mini",
    tools=[tavily_search, transfer_to_math],
    prompt="""연구 전문가입니다. 
    검색 결과에서 숫자 데이터를 찾고 계산이 필요하면 수학 전문가에게 전달하세요.""",
    name="research_agent"
)

math_agent = create_react_agent(
    model="openai:gpt-4o-mini",
    tools=[add, multiply, divide, transfer_to_research],
    prompt="""수학 전문가입니다. 
    계산에 필요한 데이터가 없으면 연구 전문가에게 전달하세요.""",
    name="math_agent"
)

# 스웜 생성
checkpointer = InMemorySaver()
swarm = create_swarm(
    agents=[research_agent, math_agent],
    default_active_agent="research_agent"
).compile(checkpointer=checkpointer)

# 실행
config = {"configurable": {"thread_id": "1"}}
result = swarm.invoke({
    "messages": [{
        "role": "user", 
        "content": "한국의 인구를 찾아서, 인구 수에 2를 곱해주세요."
    }]
}, config)
```

---

## **3. 계층적 아키텍처 (Hierarchical)**

계층적 아키텍처에서는 감독자의 감독자를 정의하여 멀티에이전트 시스템을 구성할 수 있습니다. 이는 감독자 아키텍처의 일반화로, 더 복잡한 제어 흐름을 가능하게 합니다.

```python
from langgraph_supervisor import create_supervisor
from langgraph_swarm import create_swarm, create_handoff_tool

# 하위 팀 1: 연구 팀 (Swarm)
research_handoff = create_handoff_tool(
    agent_name="data_analyst",
    description="상세 데이터 분석이 필요할 때"
)

basic_researcher = create_react_agent(
    model="openai:gpt-4o-mini",
    tools=[tavily_search, research_handoff],
    prompt="기본 연구자입니다. 웹 검색으로 정보를 수집합니다.",
    name="basic_researcher"
)

analyst_handoff = create_handoff_tool(
    agent_name="basic_researcher",
    description="기본 정보 검색이 필요할 때"
)

data_analyst = create_react_agent(
    model="openai:gpt-4o-mini",
    tools=[tavily_search, analyst_handoff],
    prompt="데이터 분석가입니다. 상세 분석과 인사이트를 제공합니다.",
    name="data_analyst"
)

research_team = create_swarm(
    agents=[basic_researcher, data_analyst],
    default_active_agent="basic_researcher"
).compile(name="research_team")

# 하위 팀 2: 계산 팀 (Swarm)
calc_handoff = create_handoff_tool(
    agent_name="advanced_calculator",
    description="복잡한 계산이 필요할 때"
)

basic_calculator = create_react_agent(
    model="openai:gpt-4o-mini",
    tools=[add, multiply, calc_handoff],
    prompt="기본 계산기입니다. 간단한 계산을 수행합니다.",
    name="basic_calculator"
)

advanced_handoff = create_handoff_tool(
    agent_name="basic_calculator",
    description="기본 계산이 필요할 때"
)

advanced_calculator = create_react_agent(
    model="openai:gpt-4o-mini",
    tools=[add, multiply, divide, advanced_handoff],
    prompt="고급 계산기입니다. 복잡한 계산을 수행합니다.",
    name="advanced_calculator"
)

calc_team = create_swarm(
    agents=[basic_calculator, advanced_calculator],
    default_active_agent="basic_calculator"
).compile(name="calc_team")

# 최상위 슈퍼바이저
top_supervisor = create_supervisor(
    agents=[research_team, calc_team],
    model=init_chat_model("openai:gpt-4o"),
    prompt="""최상위 슈퍼바이저입니다. 두 팀을 관리합니다:
    
1. research_team: 정보 검색과 데이터 분석
2. calc_team: 수학 계산

작업에 따라 적절한 팀에게 할당하세요.""",
    supervisor_name="top_supervisor"
).compile()

# 그래프 시각화
display(Image(top_supervisor.get_graph(xray=False).draw_mermaid_png()))
```

---

## **4. 고급 기능**

### (1) 작업 설명과 함께 핸드오프
```python
from langgraph.types import Send

def create_task_description_handoff_tool(
    *, agent_name: str, description: str = None
):
    name = f"transfer_to_{agent_name}"
    
    @tool(name, description=description)
    def handoff_tool(
        task_description: Annotated[
            str,
            "다음 에이전트가 수행할 작업의 상세 설명"
        ],
        state: Annotated[MessagesState, InjectedState],
    ) -> Command:
        task_message = {"role": "user", "content": task_description}
        agent_input = {**state, "messages": [task_message]}
        
        return Command(
            goto=[Send(agent_name, agent_input)],
            graph=Command.PARENT,
        )
    
    return handoff_tool
```

### (2) 메시지 포워딩
```python
from langgraph_supervisor.handoff import create_forward_message_tool

# 마지막 메시지를 직접 전달하는 도구
forwarding_tool = create_forward_message_tool("supervisor")

workflow = create_supervisor(
    [research_agent, math_agent],
    model=model,
    tools=[forwarding_tool]  # 토큰 절약 및 패러프레이징 방지
)
```

### (3) 출력 모드 설정
```python
# 전체 대화 기록
workflow = create_supervisor(
    agents=[agent1, agent2],
    output_mode="full_history"
)

# 마지막 메시지만
workflow = create_supervisor(
    agents=[agent1, agent2],
    output_mode="last_message"
)
```

---

## **5. 실전 예제: 복합 작업 처리**

```python
# 복합 작업을 처리하는 멀티에이전트 시스템
result = top_supervisor.invoke({
    "messages": [{
        "role": "user", 
        "content": "한국과 일본의 2024년 GDP를 찾아서 비교하고, 그 차이를 계산해주세요."
    }]
})

# 스트리밍으로 실행 과정 관찰
for chunk in top_supervisor.stream(
    {"messages": [{"role": "user", "content": "..."}]},
    subgraphs=True  # 하위 그래프 실행도 표시
):
    print(chunk)
```

---

## **모범 사례 및 고려사항**

1. **에이전트 전문화**: 각 에이전트는 명확한 책임 영역을 가져야 함
2. **핸드오프 설계**: Command.PARENT를 사용하여 부모 그래프 범위에서 핸드오프 실행
3. **상태 관리**: 필요한 경우 에이전트별 private state schema 정의
4. **오류 처리**: 각 에이전트에 적절한 오류 처리 로직 구현
5. **성능 최적화**: 불필요한 에이전트 호출 최소화

---

## **결론**

LangGraph의 멀티에이전트 시스템은 복잡한 작업을 효율적으로 처리할 수 있는 강력한 프레임워크를 제공합니다. 2024년 12월에 출시된 Command 타입은 에이전트 간 통신을 더욱 유연하게 만들어, 계층적이고 동적인 멀티에이전트 아키텍처 구현을 용이하게 합니다.

적절한 패턴(Supervisor, Swarm, Hierarchical)을 선택하고, Command를 활용한 핸드오프를 구현함으로써 확장 가능하고 유지보수가 용이한 AI 시스템을 구축할 수 있습니다.

---

## **LangGraph 멀티에이전트**


**1. 에이전트란?**
  - **에이전트**는 LLM을 사용하여 애플리케이션의 제어 흐름을 결정하는 시스템

**2. 멀티 에이전트 시스템이 필요한 이유**
  - 단일 에이전트 시스템이 복잡해지면서 다음과 같은 문제가 발생:
    - **도구 과부하**: 에이전트가 너무 많은 도구를 가져 잘못된 결정을 내림
    - **컨텍스트 복잡성**: 단일 에이전트가 추적하기에 너무 복잡한 컨텍스트
    - **전문화 필요**: 플래너, 연구자, 수학 전문가 등 여러 전문 영역이 필요

**3. 멀티 에이전트 시스템의 주요 장점**
  - **모듈성**: 개별 에이전트로 분리하여 개발, 테스트, 유지보수가 용이
  - **전문화**: 특정 도메인에 초점을 맞춘 전문 에이전트 생성으로 전체 성능 향상
  - **제어**: 에이전트 간 통신을 명시적으로 제어 가능

### **1. Supervisor 패턴** 

- **특징**: 단일 슈퍼바이저 에이전트가 다른 에이전트들의 실행을 결정
- **적용**: 중앙 집중식 제어가 필요한 경우

![Supervisor 패턴](https://langchain-ai.github.io/langgraph/agents/assets/supervisor.png)

`(1) langgraph-supervisor 패키지 사용`

- **설치**

    ```bash
    pip install langgraph-supervisor 
    ```

    ```bash
    uv add langgraph-supervisor
    ```

In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_tavily import TavilySearch
from langchain_core.tools import tool
from IPython.display import Image, display

# 1. 작업자 에이전트들 생성
# 연구 에이전트

tavily_search = TavilySearch(max_results=3)

research_agent = create_react_agent(
    model="openai:gpt-4.1-mini",
    tools=[tavily_search],
    prompt="당신은 연구 전문가입니다. 웹 검색으로 정보를 찾아주세요.",
    name="research_agent"
)

# 수학 에이전트
@tool
def add(a: float, b: float) -> float:
    """두 수를 더합니다."""
    return a + b

@tool  
def multiply(a: float, b: float) -> float:
    """두 수를 곱합니다."""
    return a * b

@tool
def divide(a: float, b: float) -> float:
    """두 수를 나눕니다."""
    if b == 0:
        return "0으로 나눌 수 없습니다."
    return a / b

math_agent = create_react_agent(
    model="openai:gpt-4.1-mini",
    tools=[add, multiply, divide],
    prompt="당신은 수학 전문가입니다. 계산을 도와드립니다.",
    name="math_agent"
)

# 2. 감독자 시스템 생성 (간단한 방법: langgraph-supervisor 패키지 사용)
from langgraph_supervisor import create_supervisor
from langchain.chat_models import init_chat_model

supervisor = create_supervisor(
    model=init_chat_model("openai:gpt-4.1"),
    agents=[research_agent, math_agent],
    prompt="""
    당신은 감독자입니다. 두 명의 에이전트를 관리합니다:
    - research_agent: 정보 검색 작업을 전담 
    - math_agent: 수학 계산 작업을 전담

    전문성을 고려하여 적절한 에이전트에게 작업을 할당하세요.
    """,
).compile()
    

# 그래프 시각화
display(Image(supervisor.get_graph().draw_mermaid_png()))

In [ ]:
# 3. 실행
result = supervisor.invoke({
    "messages": [{
        "role": "user", 
        "content": "한국의 인구를 찾아서, 인구 수에 2를 곱해주세요."
    }]
})

for m in result["messages"]:
    m.pretty_print()

`(2) 커스텀 핸드오프 도구 사용`

- **핸드오프 도구**를 사용하여 에이전트 간 통신을 명시적으로 제어 (현재 에이전트에서 다음 에이전트로 이동하는 데 사용되는 도구)
- 이때, graph 인자를 **Command.PARENT**로 설정하여 부모 그래프로 돌아가기

In [ ]:
from typing import Annotated
from langchain_core.tools import tool, InjectedToolCallId
from langchain_core.messages import ToolMessage
from langgraph.prebuilt import InjectedState
from langgraph.graph import MessagesState
from langgraph.types import Command

# 핸드오프 도구 생성 함수
def create_handoff_tool(*, agent_name: str, description: str | None = None):
    """커스텀 핸드오프 도구 생성"""
    name = f"transfer_to_{agent_name}"   # 다음에 이동할 에이전트 이름 (작업 할당)
    description = description or f"Transfer to {agent_name}"   # 에이전트 이동에 대한 설명 (작업 할당)

    @tool(name, description=description)
    def handoff_tool(
        state: Annotated[MessagesState, InjectedState], 
        tool_call_id: Annotated[str, InjectedToolCallId],
    ) -> Command:
        
        # 도구 호출 메시지 생성
        tool_message = ToolMessage(
            content=f"Successfully transferred to {agent_name}",
            name=name,
            tool_call_id=tool_call_id,
        )

        # 다음 에이전트로 이동하고 메시지 업데이트
        return Command(  
            goto=agent_name,  # 다음 에이전트로 이동 (작업자 지정)
            update={"messages": state["messages"] + [tool_message]},   # 메시지 업데이트 (작업 할당)
            graph=Command.PARENT,  # 부모 그래프 영역에서 실행 (작업 영역)
        )
    return handoff_tool

In [ ]:
from langgraph.graph import MessagesState, StateGraph, START, END
from langchain.chat_models import init_chat_model
from langgraph_supervisor import create_handoff_tool


# 핸드오프 도구 (supervisor → research_agent)
transfer_to_research_agent = create_handoff_tool(
    agent_name="research_agent",
    description="정보 검색, 조사, 찾기 작업을 연구 전문가에게 할당"
)

# 핸드오프 도구 (supervisor → math_agent)
transfer_to_math_agent = create_handoff_tool(
    agent_name="math_agent", 
    description="수학 계산, 곱셈, 덧셈 작업을 수학 전문가에게 할당"
)

# 핸드오프 도구 (research_agent, math_agent → supervisor)
transfer_to_supervisor = create_handoff_tool(
    agent_name="supervisor",
    description="작업 완료 후 슈퍼바이저에게 보고"
)

# 슈퍼바이저 에이전트
supervisor = create_react_agent(
    model=init_chat_model("openai:gpt-4.1"),
    tools=[transfer_to_research_agent, transfer_to_math_agent],
    prompt="""당신은 팀 슈퍼바이저입니다. 사용자의 요청을 분석하여 적절한 전문가에게 작업을 할당하세요.

🔍 **연구 작업** (research_agent):
- 정보 검색, 조사, 찾기
- 웹 검색이 필요한 작업
- 데이터 수집

🧮 **수학 작업** (math_agent):  
- 계산, 곱셈, 덧셈, 나눗셈
- 수치 처리

사용자 요청을 분석하고 transfer_to_research_agent 또는 transfer_to_math_agent 도구를 사용하여 작업을 할당하세요.
복합 작업의 경우 먼저 정보 수집부터 시작하세요.""",
    name="supervisor"
)

# 연구 에이전트 (슈퍼바이저에게 복귀)
research_agent = create_react_agent(
    model=init_chat_model("openai:gpt-4.1-mini"),
    tools=[tavily_search, transfer_to_supervisor],
    prompt="""당신은 연구 전문가입니다. 웹 검색을 통해 정확한 정보를 찾아 제공하세요.

작업을 완료한 후에는 반드시 transfer_to_supervisor 도구를 사용해서 슈퍼바이저에게 결과를 보고하세요.""",
    name="research_agent"
)

# 수학 에이전트 (슈퍼바이저에게 복귀)
math_agent = create_react_agent(
    model=init_chat_model("openai:gpt-4.1-mini"),
    tools=[add, multiply, divide, transfer_to_supervisor],
    prompt="""당신은 수학 전문가입니다. 정확한 계산을 수행하세요.

작업을 완료한 후에는 반드시 transfer_to_supervisor 도구를 사용해서 슈퍼바이저에게 결과를 보고하세요.""",
    name="math_agent"
)

# 그래프 구성
supervisor_graph = (
    StateGraph(MessagesState)
    .add_node("supervisor", supervisor)
    .add_node("research_agent", research_agent)
    .add_node("math_agent", math_agent)
    .add_edge(START, "supervisor")  # 항상 슈퍼바이저부터 시작
    .compile()
)

In [ ]:
# 실행
result = supervisor_graph.invoke({
    "messages": [{
        "role": "user", 
        "content": "한국의 인구를 찾아서, 인구 수에 2를 곱해주세요."
    }]
})

for m in result["messages"]:
    m.pretty_print()

### **2. Swarm 패턴** 

- **특징**: 분산형, 에이전트 간 자율적 협력
- **설치**
    - langgraph-swarm 패키지 사용

    ```bash
    pip install langgraph-swarm 
    ```

    ```bash
    uv add langgraph-swarm
    ```

![Swarm 패턴](https://langchain-ai.github.io/langgraph/agents/assets/swarm.png)

In [ ]:
from langgraph_swarm import create_swarm, create_handoff_tool

# 핸드오프 도구 생성
transfer_to_math_agent = create_handoff_tool(
    agent_name="math_agent",
    description="수학 계산이 필요할 때 수학 전문가에게 전달합니다."
)

transfer_to_research_agent = create_handoff_tool(
    agent_name="research_agent", 
    description="정보 검색이나 조사가 필요할 때 연구 전문가에게 전달합니다."
)

# 연구 에이전트 (핸드오프 도구 포함)
research_agent = create_react_agent(
    model=init_chat_model("openai:gpt-4.1-mini"),
    tools=[tavily_search, transfer_to_math_agent],
    prompt="""당신은 연구 전문가입니다. 웹 검색을 통해 정보를 찾아 제공합니다.
    
만약 검색 결과에서 숫자 데이터를 찾았고 사용자가 계산을 요청한다면, 
transfer_to_math_agent 도구를 사용해서 수학 전문가에게 작업을 전달하세요.""",
    name="research_agent"
)

# 수학 에이전트 (핸드오프 도구 포함)
math_agent = create_react_agent(
    model=init_chat_model("openai:gpt-4.1-mini"),
    tools=[add, multiply, divide, transfer_to_research_agent],
    prompt="""당신은 수학 전문가입니다. 정확한 계산을 수행합니다.
    
만약 계산에 필요한 데이터가 없거나 추가 정보가 필요하다면,
transfer_to_research_agent 도구를 사용해서 연구 전문가에게 작업을 전달하세요.""",
    name="math_agent"
)

# 스웜 생성
swarm = create_swarm(
    agents=[research_agent, math_agent],
    default_active_agent="research_agent"  # 기본적으로 연구 에이전트가 먼저 시작
).compile()

In [ ]:
# 실행
result = swarm.invoke({
    "messages": [{
        "role": "user", 
        "content": "한국의 인구를 찾아서, 인구 수에 2를 곱해주세요."
    }]
})

for m in result["messages"]:
    m.pretty_print()

### **3. 계층적 아키텍처 (Supervisor + Swarm 조합)** 

- **특징**: 슈퍼바이저와 스웜을 조합하여 복잡한 작업 흐름 구현
- **적용**: 대규모 시스템에서 에이전트 팀들을 관리할 때 (상위 슈퍼바이저가 하위 스웜들을 관리하는 계층적 시스템)

In [ ]:
from langgraph_supervisor import create_supervisor
from langgraph_swarm import create_swarm, create_handoff_tool
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model

# 하위 스웜 1: 정보 수집 팀
research_handoff = create_handoff_tool(
    agent_name="data_analyst",
    description="상세한 데이터 분석이 필요할 때 데이터 분석가에게 전달"
)

basic_researcher = create_react_agent(
    model=init_chat_model("openai:gpt-4.1-mini"),
    tools=[tavily_search, research_handoff],
    prompt="기본 연구자. 웹 검색으로 정보 수집. 상세 분석이 필요하면 데이터 분석가에게 전달.",
    name="basic_researcher"
)

analyst_handoff = create_handoff_tool(
    agent_name="basic_researcher", 
    description="기본 정보 검색이 필요할 때 기본 연구자에게 전달"
)

data_analyst = create_react_agent(
    model=init_chat_model("openai:gpt-4.1-mini"),
    tools=[tavily_search, analyst_handoff],
    prompt="데이터 분석가. 상세한 분석과 인사이트 제공. 기본 검색이 필요하면 기본 연구자에게 전달.",
    name="data_analyst"
)

research_swarm = create_swarm(
    agents=[basic_researcher, data_analyst],
    default_active_agent="basic_researcher"
).compile(name="research_swarm")

# 하위 스웜 2: 계산 팀  
calc_handoff = create_handoff_tool(
    agent_name="advanced_calculator",
    description="복잡한 계산이 필요할 때 고급 계산기에게 전달"
)

basic_calculator = create_react_agent(
    model=init_chat_model("openai:gpt-4.1-mini"),
    tools=[add, multiply, calc_handoff],
    prompt="기본 계산기. 간단한 덧셈과 곱셈 수행. 복잡한 계산은 고급 계산기에게 전달.",
    name="basic_calculator"
)

advanced_handoff = create_handoff_tool(
    agent_name="basic_calculator",
    description="기본 계산이 필요할 때 기본 계산기에게 전달"
)

advanced_calculator = create_react_agent(
    model=init_chat_model("openai:gpt-4.1-mini"),
    tools=[add, multiply, divide, advanced_handoff],
    prompt="고급 계산기. 복잡한 계산 수행. 기본 계산은 기본 계산기에게 전달.",
    name="advanced_calculator"
)

calc_swarm = create_swarm(
    agents=[basic_calculator, advanced_calculator],
    default_active_agent="basic_calculator"
).compile(name="calc_swarm")

# 최상위 슈퍼바이저
top_supervisor = create_supervisor(
    agents=[research_swarm, calc_swarm],
    model=init_chat_model("openai:gpt-4.1"),
    prompt="""최상위 슈퍼바이저입니다. 두 개의 전문 팀을 관리합니다:

1. research_swarm: 정보 검색과 데이터 분석 담당
2. calc_swarm: 수학 계산 담당

작업의 성격에 따라 적절한 팀에게 할당하세요."""
).compile(name="top_supervisor")


# 그래프 시각화
display(Image(top_supervisor.get_graph(xray=False).draw_mermaid_png()))

In [ ]:
# 실행
result = top_supervisor.invoke({
    "messages": [{
        "role": "user", 
        "content": "한국의 인구를 찾아서, 인구 수에 2를 곱해주세요."
    }]
})

for m in result["messages"]:
    m.pretty_print()